# **SQL Database with Python**

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('kolesa_market_v2.db')
cursor = conn.cursor()

cursor.execute("DROP TABLE IF EXISTS cars")
cursor.execute("DROP TABLE IF EXISTS brands")
cursor.execute("DROP TABLE IF EXISTS cities")

# Table 1: Brands
cursor.execute("CREATE TABLE brands (id INTEGER PRIMARY KEY, name TEXT UNIQUE)")

# Table 2: Cities
cursor.execute("CREATE TABLE cities (id INTEGER PRIMARY KEY, name TEXT UNIQUE)")

# Table 3: Cars (with relationship FOREIGN KEY)
cursor.execute("""
CREATE TABLE cars (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT,
    brand_id INTEGER,
    city_id INTEGER,
    year INTEGER,
    price INTEGER,
    mileage INTEGER,
    is_bargain TEXT,
    FOREIGN KEY (brand_id) REFERENCES brands (id),
    FOREIGN KEY (city_id) REFERENCES cities (id)
)
""")

# 3. Insert cleaned data into the database using Python
df = pd.read_excel('Kolesa_Cleaned_Version.xlsx')

for b_name in df['brand'].unique():
    cursor.execute("INSERT OR IGNORE INTO brands (name) VALUES (?)", (b_name,))
for c_name in df['city'].unique():
    cursor.execute("INSERT OR IGNORE INTO cities (name) VALUES (?)", (c_name,))

# Mapping for relations
brand_map = {name: id for id, name in cursor.execute("SELECT id, name FROM brands")}
city_map = {name: id for id, name in cursor.execute("SELECT id, name FROM cities")}

# INSERT
for _, row in df.iterrows():
    cursor.execute("""
        INSERT INTO cars (title, brand_id, city_id, year, price, mileage, is_bargain)
        VALUES (?, ?, ?, ?, ?, ?, ?)
    """, (row['title'], brand_map[row['brand']], city_map[row['city']],
          row['year'], row['price'], row['mileage'], row['is_bargain']))

conn.commit()
print("--- Database created and populated ---")

# SELECT
print("\n[SELECT] Top 3 cars:")
print(pd.read_sql("""
    SELECT b.name as Brand, c.title, ci.name as City, c.price
    FROM cars c
    JOIN brands b ON c.brand_id = b.id
    JOIN cities ci ON c.city_id = ci.id
    ORDER BY c.price DESC LIMIT 3
""", conn))

# UPDATE
cursor.execute("UPDATE cars SET is_bargain = 'Premium' WHERE price > 60000000")
conn.commit()
print(f"\n[UPDATE] Rows updated: {cursor.rowcount}")

# DELETE
cursor.execute("DELETE FROM cars WHERE year < 1990")
conn.commit()
print(f"[DELETE] Rows deleted: {cursor.rowcount}")

--- Database created and populated ---

[SELECT] Top 3 cars:
           Brand                   title    City      price
0    Rolls-royce    Rolls-royce cullinan  Астана  450000000
1        Porsche      Porsche 911 gt3 rs  Алматы  310000000
2  Mercedes-benz  Mercedes-benz g 63 amg  Алматы  285000000

[UPDATE] Rows updated: 977
[DELETE] Rows deleted: 0


# **Data Analysis (SQL Queries)**

In [ ]:
# 1. Average Price by Brand (Aggregation + JOIN + GROUP BY + ORDER BY)
# This query calculates which brands are the most expensive on average.
query1 = """
SELECT b.name as Brand, ROUND(AVG(c.price), 0) as Avg_Price, COUNT(c.id) as Total_Cars
FROM cars c
JOIN brands b ON c.brand_id = b.id
GROUP BY b.name
HAVING Total_Cars > 5
ORDER BY Avg_Price DESC
LIMIT 10
"""
print("\n[Query 1] Average Price by Brand (Top 10):")
print(pd.read_sql(query1, conn))


[Query 1] Average Price by Brand (Top 10):
              Brand    Avg_Price  Total_Cars
0       Rolls-royce  165666667.0           9
1           Bentley  142915385.0          13
2       Lamborghini  141428571.0           7
3  Mercedes-maybach  140271646.0          16
4          Maserati  123566667.0           6
5           Porsche  105389051.0          56
6     Mercedes-benz  104797011.0         287
7              Land  101312231.0         104
8            Rivian   88241000.0           8
9          Cadillac   88186316.0          38


**Insight: Shows the premium segment of the market and brand positioning.**

In [ ]:
# 2. Count of Items per Location (Aggregation + JOIN + GROUP BY + ORDER BY)
# This query identifies which cities have the highest number of listings.
query2 = """
SELECT ci.name as City, COUNT(c.id) as Listing_Count
FROM cars c
JOIN cities ci ON c.city_id = ci.id
GROUP BY ci.name
ORDER BY Listing_Count DESC
"""
print("\n[Query 2] Distribution of Listings by City:")
print(pd.read_sql(query2, conn))


[Query 2] Distribution of Listings by City:
                City  Listing_Count
0             Алматы            518
1             Астана            241
2            Шымкент             72
3          Караганда             24
4             Атырау             20
5   Усть-каменогорск             19
6           Костанай             18
7              Актау             18
8             Актобе             16
9           Павлодар             15
10             Семей             11
11           Уральск             10
12             Тараз             10
13          Кокшетау              8
14         Кызылорда              5
15         Экибастуз              3
16     Петропавловск              3
17           Щучинск              2
18              Аксу              2
19       Талдыкорган              1
20            Москва              1
21           Жетысай              1
22          Жанаозен              1


**Insight: Helps determine which regional markets are the most active (e.g., Almaty vs Astana).**

In [ ]:
# 3. Top 10 Most Expensive Luxury Items (Filtering + Sorting)
# Using WHERE to filter for specific high-end conditions and ORDER BY for ranking.
query3 = """
SELECT title, price, year, mileage
FROM cars
WHERE price > 50000000 AND mileage < 20000
ORDER BY price DESC
LIMIT 10
"""
print("\n[Query 3] Top 10 High-End Luxury Listings (Price > 50M & Low Mileage):")
print(pd.read_sql(query3, conn))


[Query 3] Top 10 High-End Luxury Listings (Price > 50M & Low Mileage):
                    title      price  year  mileage
0    Rolls-royce cullinan  450000000  2024     2600
1      Porsche 911 gt3 rs  310000000  2018        0
2  Mercedes-benz g 63 amg  285000000  2024        0
3  Bentley continental gt  280000000  2017        0
4      Ferrari purosangue  243000000  2022        0
5  Mercedes-benz g 63 amg  238000000  2024        0
6    Rolls-royce cullinan  230000000  2018      500
7        Bentley bentayga  229900000  2015        0
8         Porsche 911 gt3  216127800  2024        0
9        Bentley bentayga  214200000  2015        0


**Insight: Identifies "unicorn" listings—extremely expensive cars in nearly new condition.**


**Insight: Helps to see which days are most active for posting ads.**

In [1]:
# 4. Bargain Hunting: Average Discount by City (Filtering + Aggregation)
# Analyzes where the "Bargain" deals are most concentrated.
query4 = """
SELECT ci.name as City, COUNT(c.id) as Bargain_Count, ROUND(AVG(c.price_diff), 0) as Avg_Savings
FROM cars c
JOIN cities ci ON c.city_id = ci.id
WHERE c.is_bargain = 'Good Price' OR c.is_bargain = 'HOT DEAL'
GROUP BY ci.name
ORDER BY Avg_Savings DESC
"""
print("\nMarket Efficiency: Best cities to find 'Bargain' deals:")
try:
    print(pd.read_sql(query4, conn))
except:
    print("Column 'price_diff' not found. Check Step 4 schema.")


Market Efficiency: Best cities to find 'Bargain' deals:
Column 'price_diff' not found. Check Step 4 schema.


**Insight: Highlights cities where sellers are more likely to list cars below market average.**


In [ ]:
# 5. Inventory Age Analysis (Trend/Categorization + Grouping)
# Shows how many cars are available from different eras.
query5 = """
SELECT
    CASE
        WHEN year >= 2020 THEN 'New (2020-2026)'
        WHEN year >= 2010 THEN 'Modern (2010-2019)'
        WHEN year >= 2000 THEN 'Older (2000-2009)'
        ELSE 'Vintage (<2000)'
    END as Era,
    COUNT(*) as Car_Count,
    AVG(price) as Avg_Era_Price
FROM cars
GROUP BY Era
ORDER BY year DESC
"""
print("\nInventory Distribution by Era:")
print(pd.read_sql(query5, conn))


Inventory Distribution by Era:
                  Era  Car_Count  Avg_Era_Price
0     New (2020-2026)        635   8.982444e+07
1  Modern (2010-2019)        375   9.543657e+07
2   Older (2000-2009)          9   1.039667e+08


**Insight: Reveals the "freshness" of the market inventory and how value drops with age.**

In [ ]:
conn.close()